# 04 - async_execution 加速

如果某些 task 之间没有依赖,可以并行跑。
CrewAI 用 `Task(async_execution=True)` 标记,让 manager 调度时并行启动。

本 notebook 跑 3 个独立 task(3 种语言的问候),对比:

- 全部 sequential
- 全部 `async_execution=True`

总耗时差异会很明显(几乎 3 倍)。

**注意**:本 notebook 会发 6 次 LLM 调用,合计约 30-60 秒。


In [4]:
import time
from alphaquant.infrastructure.llm import get_llm
from crewai import Agent, Task, Crew, Process

llm = get_llm(temperature=0.1)

spanish_agent = Agent(
    role="西语翻译",
    goal="用西语问候",
    backstory="你用西语说你好。",
    tools=[],
    llm=llm,
)
french_agent = Agent(
    role="法语翻译",
    goal="用法语问候",
    backstory="你用法语说你好。",
    tools=[],
    llm=llm,
)
japanese_agent = Agent(
    role="日语翻译",
    goal="用日语问候",
    backstory="你用日语说你好。",
    tools=[],
    llm=llm,
)


In [5]:
seq_tasks = [
    Task(description="用西语说'你好'", expected_output="一个西语问候", agent=spanish_agent),
    Task(description="用法语说'你好'", expected_output="一个法语问候", agent=french_agent),
    Task(description="用日语说'你好'", expected_output="一个日语问候", agent=japanese_agent),
]
seq_crew = Crew(
    agents=[spanish_agent, french_agent, japanese_agent],
    tasks=seq_tasks,
    process=Process.sequential,
    verbose=False,
)

t0 = time.time()
seq_crew.kickoff(inputs={})
seq_t = time.time() - t0
print(f"Sequential 总耗时: {seq_t:.2f}s")


Sequential 总耗时: 15.83s


In [ ]:
# 关键约束:CrewAI 0.203 的 Crew 模型校验规则 (`crew.py:435-454`)
# 从尾部反向数连续 async task,遇 sync 就停,计数必须 ≤ 1。
# 所以 3 个全 async 会失败,必须最后 1 个 sync。
async_tasks = [
    Task(description="用西语说'你好'", expected_output="一个西语问候", agent=spanish_agent, async_execution=True),
    Task(description="用法语说'你好'", expected_output="一个法语问候", agent=french_agent, async_execution=True),
    Task(description="用日语说'你好'", expected_output="一个日语问候", agent=japanese_agent),  # 最后 1 个必须 sync
]
async_crew = Crew(
    agents=[spanish_agent, french_agent, japanese_agent],
    tasks=async_tasks,
    process=Process.sequential,
    verbose=False,
)

t0 = time.time()
async_crew.kickoff(inputs={})
async_t = time.time() - t0
print(f"Async 总耗时: {async_t:.2f}s")
print(f"加速比: {seq_t / async_t:.2f}x")


## 为什么前 2 个 async,最后 1 个 sync?

CrewAI 0.203 的 `Crew` 模型有这个硬约束 (`crew.py:435-454`):

```python
final_async_task_count = 0
for task in reversed(self.tasks):
    if task.async_execution:
        final_async_task_count += 1
    else:
        break  # 遇 sync 就停
if final_async_task_count > 1:
    raise PydanticCustomError('async_task_count', ...)
```

**解读**:反向遍历 task 列表,从尾部数连续 async 的数量,
遇到第一个 sync 就停。计数必须 ≤ 1。

实测:

- `[async, async, sync]` ✓ 尾部 0 个 (最后是 sync 立刻 break)
- `[async, async, async]` ✗ 尾部 3 个
- `[sync, sync, async]` ✓ 尾部 1 个
- `[sync, async, async]` ✗ 尾部 2 个

所以本 notebook 把日语 task 设成 sync —— 西语 + 法语并行跑,
完成后日语作 barrier。

## 跟生产代码的对应

`AnalysisCrew` 用 `_ASYNC_TASK_INDICES = {0, 1, 2, 3, 4, 5, 6}`:
前 7 个 task (数据 0-3 + 分析 4-6) 都是 async,第 8 个 (report_writer) sync。
按上面的规则,尾部 0 个连续 async → 通过校验。

在 `crews/analysis_crew.py` 的 `_build_tasks()` 中:

```python
async_execution=idx in self._ASYNC_TASK_INDICES,
```

`report_writer` 必须 sync 的原因:

1. 它需要 `tasks[4]`、`tasks[5]`、`tasks[6]` 的输出作为 context,manager 会
   等 context 准备好才启动它(显式 `context=[]` 可以解决)
2. 即便 context 注入 OK,如果数据 task 还没跑完,manager 调度时可能让它
   跟数据 task 抢资源,反而变慢

`_ASYNC_TASK_INDICES` 的设计原则:

- 没有数据依赖的 task 标 async (尾部会有 1+ 个 sync 作 barrier)
- 报告写作者必须最后跑 (用 `context=[tasks[4], tasks[5], tasks[6]]` 显式注入)
- 显式控制并行边界,比"全部 async"更稳定
